# 🎓 SWU Open House — AI Hand Gesture Classifier (Dev 1)
### ภาควิชาวิศวกรรมคอมพิวเตอร์ มหาวิทยาลัยศรีนครินทรวิโรฒ (CPE SWU)

สมุดบันทึก (Notebook) สำหรับเก็บข้อมูลพิกัดมือ 21 จุด (Landmarks) และเทรนโมเดลจำแนกท่าทาง:
- 🖐️ `OPEN_PALM`: ท่าแบมือค้าง 1.5 วินาที เพื่อสั่งนับถอยหลังถ่ายภาพ
- ✊ `FIST`: ท่ากำมือในหน้าพรีวิว เพื่อสั่งถ่ายใหม่ (Retake)
- 🔄 `NONE`: ท่าทางอื่นๆ / ไม่มีมือ

**เป้าหมาย**: ความแม่นยำ $\ge 95\%$ และ Export ค่าน้ำหนักโมเดล (`gesture_weights.json`) สำหรับใช้บนหน้าเว็บ Kiosk

## 1. ติดตั้ง Dependencies ใน Colab
*(TensorFlow, Scikit-learn, Pandas, NumPy มีอยู่ใน Colab อยู่แล้ว ติดตั้งแค่ `mediapipe` เพิ่ม)*

In [ ]:
!pip install mediapipe scikit-learn pandas numpy

## 2. ฟังก์ชัน Normalization พิกัด 21 จุด (63 Features)
ปรับจุดอ้างอิงให้ข้อมือ (จุดที่ 0) อยู่ที่พิกัด $(0, 0, 0)$ และปรับ Scale ด้วยระยะทางระหว่างข้อมือถึงโคนนิ้วกลาง (จุดที่ 9) เพื่อให้โมเดลไม่ขึ้นกับขนาดมือและระยะใกล้ไกลของกล้อง

In [ ]:
import math
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

def normalize_landmarks_array(coords_21x3):
    """
    coords_21x3: list or ndarray of shape (21, 3)
    Returns: flat list of 63 normalized coordinates
    """
    wrist = coords_21x3[0]
    base_x, base_y, base_z = wrist[0], wrist[1], wrist[2]
    
    rel_pts = []
    for pt in coords_21x3:
        rel_pts.append((pt[0] - base_x, pt[1] - base_y, pt[2] - base_z))
        
    mcp_mid = rel_pts[9]
    scale = math.sqrt(mcp_mid[0]**2 + mcp_mid[1]**2 + mcp_mid[2]**2)
    if scale < 1e-6:
        scale = 1.0
        
    flat = []
    for x, y, z in rel_pts:
        flat.extend([round(x / scale, 6), round(y / scale, 6), round(z / scale, 6)])
    return flat

print("✓ Normalization function ready.")

## 3. สร้างข้อมูลตัวอย่าง (Synthetic) หรืออัปโหลดไฟล์ `gestures.csv`
หากมีไฟล์ `gestures.csv` จากเครื่องคอมพิวเตอร์ ให้รันเซลล์ด้านล่างเพื่ออัปโหลด หรือรันเพื่อจำลองข้อมูลทดสอบเบื้องต้น

In [ ]:
# เลือกว่าจะอัปโหลด gestures.csv หรือสร้างตัวอย่าง Synthetic
UPLOAD_EXISTING_CSV = False

if UPLOAD_EXISTING_CSV:
    print("กรุณาเลือกไฟล์ gestures.csv จากเครื่องคอมพิวเตอร์ของคุณ:")
    uploaded = files.upload()
    csv_filename = list(uploaded.keys())[0]
    df = pd.read_csv(csv_filename)
    print(f"✓ โหลด {csv_filename} เรียบร้อย ({len(df)} แถว)")
else:
    # จำลองชุดข้อมูลสังเคราะห์ 300 ตัวอย่างสำหรับการทดลองเทรน
    np.random.seed(42)
    data = []
    cols = ["label"] + [f"{ax}{i}" for i in range(21) for ax in ("x", "y", "z")]
    
    for _ in range(120):
        # OPEN_PALM: ปลายนิ้วเหยียดสูง
        pts = np.random.randn(21, 3) * 0.05
        for tip in [8, 12, 16, 20]:
            pts[tip][1] -= 0.6  # ยื่นขึ้นด้านบน
        norm = normalize_landmarks_array(pts)
        data.append(["OPEN_PALM"] + norm)
        
    for _ in range(120):
        # FIST: ปลายนิ้วงอเข้าหาฝ่ามือ
        pts = np.random.randn(21, 3) * 0.05
        for tip in [8, 12, 16, 20]:
            pts[tip][1] += 0.2  # งอลงล่าง
        norm = normalize_landmarks_array(pts)
        data.append(["FIST"] + norm)
        
    for _ in range(80):
        # NONE: ท่าทางสุ่ม
        pts = np.random.randn(21, 3) * 0.2
        norm = normalize_landmarks_array(pts)
        data.append(["NONE"] + norm)
        
    df = pd.DataFrame(data, columns=cols)
    print(f"✓ สร้าง Synthetic Dataset เรียบร้อย: {len(df)} แถว")
    print("⚠️ ข้อมูลจำลองใช้ทดสอบขั้นตอนเท่านั้น ห้ามนำผลหรือโมเดลไปใช้จริง")

print("\nการกระจายตัวของคลาส:")
print(df["label"].value_counts())

## 4. เทรนโมเดล MLP Neural Network & ประเมินผลความแม่นยำ (Target $\ge 95\%$)

In [ ]:
feature_cols = [c for c in df.columns if c != "label"]
X = df[feature_cols].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    max_iter=500,
    random_state=42,
    early_stopping=True
)

clf.fit(X_train, y_train)

train_acc = clf.score(X_train, y_train) * 100
test_acc = clf.score(X_test, y_test) * 100

print(f"🎯 ความแม่นยำ Train Set: {train_acc:.2f}%")
print(f"🎯 ความแม่นยำ Test Set : {test_acc:.2f}%")

y_pred = clf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

if not UPLOAD_EXISTING_CSV:
    print("\n⚠️ เทรนจากข้อมูลจำลอง: ค่าความแม่นยำนี้ไม่ใช่ผลจริง ยังไม่ผ่านเกณฑ์")
elif test_acc >= 95.0:
    print("\n✅ ผ่านเกณฑ์ Acceptance Criteria (Accuracy >= 95%)")
else:
    print("\n⚠️ ความแม่นยำยังไม่ถึง 95% แนะนำให้เพิ่มปริมาณชุดข้อมูล")

## 5. Export ค่าน้ำหนักเป็น JSON (`gesture_weights.json`) สำหรับ Kiosk Web App
ไฟล์นี้จะถูกดาวน์โหลดลงเครื่องอัตโนมัติ ให้นำไปวางที่โฟลเดอร์ `web/public/models/` ของโปรเจกต์

In [ ]:
layers = []
for idx, (w, b) in enumerate(zip(clf.coefs_, clf.intercepts_)):
    layers.append({
        "layerIndex": idx,
        "weights": np.round(w, 6).tolist(),
        "biases": np.round(b, 6).tolist(),
        "inputDim": int(w.shape[0]),
        "outputDim": int(w.shape[1]),
    })

export_payload = {
    "modelType": "MLPClassifier",
    "exportedAt": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "classes": [str(c) for c in clf.classes_],
    "inputDimension": 63,
    "layers": layers,
    "activation": clf.activation,
    "outputActivation": clf.out_activation_,
}

json_filename = "gesture_weights.json"
if not UPLOAD_EXISTING_CSV:
    print("⚠️ ไฟล์นี้เทรนจากข้อมูลจำลอง ห้ามนำไปวางใน web/public/models/")
with open(json_filename, "w", encoding="utf-8") as f:
    json.dump(export_payload, f, indent=2)

print(f"✓ บันทึก {json_filename} เรียบร้อย")
if IN_COLAB and files:
    print("กำลังดาวน์โหลดไฟล์...")
    files.download(json_filename)
else:
    print(f"✓ บันทึกไฟล์ในเครื่องเรียบร้อย: {json_filename}")